In [5]:
!pip install -q -U langchain langchain-openai

In [11]:
from google.colab import userdata
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    temperature=0.3
)

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE: You are the friendly assistant for Sweet Crumbs Home Bakery.

FACTS: Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
       Orders need 24 hours' notice. All cakes have eggless versions.

RULES: Never promise a delivery time or place.
       If it isn't in FACTS, say "I'll check with the owner."
       Work out totals carefully.

FORMAT: Under 50 words, friendly, English or Telugu matching the customer.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Hi Ravi! 😊 Yes, we offer egg‑less versions of all our cakes, including plum, chocolate, and cupcakes. Let us know what you’d like, and we’ll get it ready for you!
Your name is Ravi. 😊


In [13]:
history

[('human', "Hi, I'm Ravi. Do you make eggless cakes?"),
 ('ai',
  'Hi Ravi! 😊 Yes, we offer egg‑less versions of all our cakes, including plum, chocolate, and cupcakes. Let us know what you’d like, and we’ll get it ready for you!'),
 ('human', "What's my name?"),
 ('ai', 'Your name is Ravi. 😊')]

In [14]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

In [15]:
run_tests()

PASS | How much is a plum cake? → A plum cake costs Rs 500. Let me know if you’d like to add anything el
FAIL | Do you make eggless cakes? → Yes! All our cakes—including plum, chocolate, and cupcakes—are availab
FAIL | Can I get a cake today evening? → I’ll check with the owner and get back to you shortly!
PASS | Do you deliver to Kakinada? → I’ll check with the owner and get back to you!
PASS | What is the price of 6 cupcakes? → Sure! Six cupcakes cost ₹ 480 (₹ 80 each). Let me know if you'd like t

3/5 passed (60%)
